# 半监督 GNN 空间转录组细胞类型注释
## 转录本级别流水线（不依赖细胞分割）

**数据集**：GSE264393（健康肾脏 snRNA-seq）+ GSE264334（IgAN 肾脏 Xenium）

**缓存文件说明**：
| 文件 | 内容 | 节省时间 |
|---|---|---|
| `scrna_export/flex_expr.npy` | scRNA 表达矩阵 | 5-10 分钟 |
| `graph_spot/graph_*.pt` | 联合图结构 | 30-60 分钟 |
| `models/{name}_best.pt` | 训练权重 | 数小时 |
| `predictions/spot_probas_all.pkl` | 所有节点预测概率 | 5-10 分钟 |

In [1]:
# ============================================================
# Cell 1: 环境配置
# ============================================================
import os, sys, warnings, pickle, json
import numpy as np
import pandas as pd
import torch
warnings.filterwarnings('ignore')

PATHS = {
    'raw': {
        'transcripts': './data/raw/xenium/GSE264334_RAW/transcripts.csv.gz',
        'cells_csv'  : './data/raw/xenium/GSE264334_RAW/cells.csv.gz',
    },
    'cache': {
        'scrna_rds'    : './data/cache/kidney_scrna_processed.rds',
        'xenium_rds'   : './data/cache/kidney_xenium_processed.rds',
        'scrna_export' : './data/cache/scrna_export/',   # numpy 导出缓存
        'graph'        : './data/cache/graph_spot/',
    },
    'output': {
        'models'     : './results/models/',
        'predictions': './results/predictions/',
        'plots'      : './plots/',
    },
}
for d in [*PATHS['output'].values(),
          PATHS['cache']['graph'],
          PATHS['cache']['scrna_export']]:
    os.makedirs(d, exist_ok=True)

PARAMS = {
    'bin_size'       : 5,
    'qv_threshold'   : 20,
    'min_transcripts': 1,
    'k_feat'         : 15,
    'k_spatial'      : 10,
    'val_ratio'      : 0.2,
    'hidden_dim'     : 256,
    'proj_dim'       : 512,
    'dropout'        : 0.5,
    'n_epochs'       : 500,
    'lr'             : 1e-3,
    'weight_decay'   : 5e-4,
    'patience'       : 40,
    'lr_factor'      : 0.5,
    'lr_patience'    : 15,
    'min_lr'         : 1e-5,
    'max_grad_norm'  : 1.0,
    'warmup_epochs'  : 30,
    'lambda_mmd'     : 0.1,
    'lambda_ent'     : 0.01,
    'lambda_pl'      : 0.3,
    'pl_threshold'   : 0.90,
    'pl_update_freq' : 10,
    'device'         : 'cpu',
    'seed'           : 42,
}

# ── 强制重建开关（改为 True 可忽略对应缓存）──────────────────────
FORCE = {
    'scrna_export': False,   # Cell 2: 重新从 R 导出 scRNA 矩阵
    'graph'       : False,   # Cell 3: 重新构建 spot 图
    'train'       : False,   # Cell 4b: 重新训练模型
    'eval'        : False,   # Cell 5: 重新推断预测概率
}

from utils import set_seed
set_seed(PARAMS['seed'])
from gpu_utils import list_gpus, select_gpu
list_gpus(show_processes=True)
PARAMS['device'] = select_gpu('auto', min_free_gb=8.0)

# ── 缓存状态一览 ──────────────────────────────────────────────
cache_tag  = (f"kf{PARAMS['k_feat']}_ks{PARAMS['k_spatial']}"
              f"_bin{PARAMS['bin_size']}_val{PARAMS['val_ratio']}")
GRAPH_FILE  = os.path.join(PATHS['cache']['graph'], f'graph_{cache_tag}.pt')
SCALER_FILE = os.path.join(PATHS['cache']['graph'], f'scaler_{cache_tag}.pkl')
COORDS_FILE = os.path.join(PATHS['cache']['graph'], f'spot_coords_{cache_tag}.npy')
RESULTS_SUMMARY_FILE = os.path.join(PATHS['output']['models'], 'gnn_results_summary.pkl')
SPOT_PROBAS_FILE     = os.path.join(PATHS['output']['predictions'], 'spot_probas_all.pkl')
EXPORT_FILES = {
    'flex_expr'         : os.path.join(PATHS['cache']['scrna_export'], 'flex_expr.npy'),
    'flex_labels'       : os.path.join(PATHS['cache']['scrna_export'], 'flex_labels.npy'),
    'cell_types'        : os.path.join(PATHS['cache']['scrna_export'], 'cell_types.json'),
    'xenium_panel_genes': os.path.join(PATHS['cache']['scrna_export'], 'xenium_panel_genes.json'),
}

print('\n=== 缓存状态 ===')
checks = [
    ('scRNA numpy 导出', all(os.path.exists(f) for f in EXPORT_FILES.values())),
    ('spot 图',          all(os.path.exists(p) for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE])),
    ('训练权重 (GCN)',   os.path.exists(os.path.join(PATHS['output']['models'], 'GCN_best.pt'))),
    ('训练权重 (SAGE)',  os.path.exists(os.path.join(PATHS['output']['models'], 'GraphSAGE_best.pt'))),
    ('训练权重 (GAT)',   os.path.exists(os.path.join(PATHS['output']['models'], 'GAT_best.pt'))),
    ('预测概率',         os.path.exists(SPOT_PROBAS_FILE)),
]
for name, hit in checks:
    print(f'  {"[HIT]" if hit else "[MISS]"} {name}')
print(f'\n训练设备: {PARAMS["device"]}')


┌──────────────────────────────────────────────────────────────────────┐
│ ID  型号                               空闲     已用     总计    占用 状态
├──────────────────────────────────────────────────────────────────────┤
│ [0]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [1]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [2]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [3]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [4]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [5]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [6]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
│ [7]  NVIDIA GeForce RTX 3080       19.3GB    0.4GB   19.7GB     2%  空闲 ✓
└──────────────────────────────────────────────────────────────────────┘

  安装 pynvml+psutil 可查看占用进程
[auto] 选择 GPU 0（空闲 19.3 GB）

已选 GPU 0：NVIDIA GeForce RTX 3080
  空闲：19.3 GB / 20 GB （已用 

In [2]:
# ============================================================
# Cell 2: 导出 scRNA 数据（替代原 %%R 魔法命令）
# 缓存策略：
#   numpy 缓存存在 → 直接加载（< 10 秒）
#   不存在         → 通过 rpy2 从 .rds 导出并保存（5-10 分钟）
# ============================================================
EXPORT_OK = (all(os.path.exists(f) for f in EXPORT_FILES.values())
             and not FORCE['scrna_export'])

if EXPORT_OK:
    print('[Cache HIT] 从 numpy 缓存加载 scRNA 导出数据...')
    flex_expr          = np.load(EXPORT_FILES['flex_expr'])
    flex_labels        = np.load(EXPORT_FILES['flex_labels'])
    with open(EXPORT_FILES['cell_types']) as f:
        cell_types = json.load(f)
    with open(EXPORT_FILES['xenium_panel_genes']) as f:
        xenium_panel_genes = json.load(f)

else:
    print('[Cache MISS] 从 R 导出 scRNA 数据（约 5-10 分钟）...')
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri, default_converter
    from rpy2.robjects.conversion import localconverter
    numpy2ri.activate()

    ro.r('suppressPackageStartupMessages(library(Seurat))')
    ro.r('cat("Loading scRNA reference...\\n")')
    ro.r('scrna_obj  <- readRDS("./data/cache/kidney_scrna_processed.rds")')
    ro.r('xenium_obj <- readRDS("./data/cache/kidney_xenium_processed.rds")')
    ro.r('''
        xenium_panel_genes_r <- rownames(xenium_obj)
        common_genes <- intersect(rownames(scrna_obj), xenium_panel_genes_r)
        cat("共同基因:", length(common_genes), "\\n")

        flex_counts  <- as.matrix(
            GetAssayData(scrna_obj, layer = "counts")[common_genes, ])
        flex_expr_r  <- t(flex_counts)

        labels_raw   <- scrna_obj$cell_type
        cell_types_r <- sort(unique(labels_raw))
        label_map    <- setNames(seq_along(cell_types_r) - 1L, cell_types_r)
        flex_labels_r <- as.integer(label_map[labels_raw])
        cat("导出完成\\n")
    ''')

    flex_expr          = np.array(ro.r('flex_expr_r'),          dtype=np.float32)
    flex_labels        = np.array(ro.r('flex_labels_r'),        dtype=np.int64)
    cell_types         = list(ro.r('cell_types_r'))
    xenium_panel_genes = list(ro.r('xenium_panel_genes_r'))

    # 保存 numpy 缓存
    np.save(EXPORT_FILES['flex_expr'],   flex_expr)
    np.save(EXPORT_FILES['flex_labels'], flex_labels)
    with open(EXPORT_FILES['cell_types'], 'w') as f:
        json.dump(cell_types, f)
    with open(EXPORT_FILES['xenium_panel_genes'], 'w') as f:
        json.dump(xenium_panel_genes, f)
    print(f'✅ scRNA 导出缓存已保存 → {PATHS["cache"]["scrna_export"]}')
    print('   下次运行将直接从 numpy 加载，跳过 R 调用')

# 公共变量（后续 Cell 使用）
gene_list       = xenium_panel_genes
cell_types_list = cell_types
n_classes       = len(cell_types_list)
print(f'\nscRNA 细胞数 : {flex_expr.shape[0]:,}')
print(f'特征维度     : {flex_expr.shape[1]}')
print(f'细胞类型数   : {n_classes}')
print(f'细胞类型     : {cell_types_list}')

[Cache MISS] 从 R 导出 scRNA 数据（约 5-10 分钟）...

    an issue that caused a segfault when used with rpy2:
    https://github.com/rstudio/reticulate/pull/1188
    Make sure that you use a version of that package that includes
    the fix.
    Loading scRNA reference...


R[write to console]: Error in gzfile(file, "rb") : cannot open the connection

R[write to console]: In addition: 
R[write to console]: Warning message:

R[write to console]: In gzfile(file, "rb") :
R[write to console]: 
 
R[write to console]:  cannot open compressed file './data/cache/kidney_xenium_processed.rds', probable reason 'No such file or directory'



RRuntimeError: Error in gzfile(file, "rb") : cannot open the connection


In [ ]:
# ============================================================
# Cell 3: 加载转录本 + 构建 spot 图
# 缓存策略：已有完整缓存（Cell 1 已验证），直接加载
# FORCE['graph'] = True 可强制重建
# ============================================================
from utils_spot import (
    load_xenium_transcripts,
    bin_transcripts_to_spots,
    unified_normalize_spot,
    build_spot_graph,
)

GRAPH_OK = (all(os.path.exists(p)
                for p in [GRAPH_FILE, SCALER_FILE, COORDS_FILE])
            and not FORCE['graph'])

if GRAPH_OK:
    print('[Cache HIT] 加载 spot 图...')
    ckpt          = torch.load(GRAPH_FILE, weights_only=False)
    data          = ckpt['data']
    class_weights = ckpt['class_weights']
    split_info    = ckpt['split_info']
    with open(SCALER_FILE, 'rb') as f:
        fitted_scaler = pickle.load(f)
    spot_coords = np.load(COORDS_FILE)

else:
    print('[Cache MISS] 从转录本构建 spot 图...')

    df_tx = load_xenium_transcripts(
        PATHS['raw']['transcripts'],
        gene_list    = gene_list,
        qv_threshold = PARAMS['qv_threshold'],
    )
    spot_expr, spot_coords = bin_transcripts_to_spots(
        df_tx,
        gene_list       = gene_list,
        bin_size        = PARAMS['bin_size'],
        min_transcripts = PARAMS['min_transcripts'],
    )
    n_spots = spot_expr.shape[0]
    print(f'Spot 节点数: {n_spots:,}')
    if n_spots > 500_000:
        print(f'⚠️  节点数较多，建议增大 bin_size={PARAMS["bin_size"]} → 10')

    print('归一化...')
    scrna_norm, spot_norm, fitted_scaler = unified_normalize_spot(
        flex_expr.astype(np.float32), spot_expr)

    data, class_weights, split_info = build_spot_graph(
        scrna_norm   = scrna_norm,
        spot_norm    = spot_norm,
        spot_coords  = spot_coords,
        scrna_labels = flex_labels.astype(np.int64),
        k_feat       = PARAMS['k_feat'],
        k_spatial    = PARAMS['k_spatial'],
        val_ratio    = PARAMS['val_ratio'],
    )
    torch.save({'data': data, 'class_weights': class_weights,
                'split_info': split_info}, GRAPH_FILE)
    with open(SCALER_FILE, 'wb') as f:
        pickle.dump(fitted_scaler, f)
    np.save(COORDS_FILE, spot_coords)
    print(f'✅ 图缓存已保存 → {GRAPH_FILE}')

print(f'\n节点数    : {data.x.shape[0]:,}  (scRNA {split_info["n_scrna"]:,} + spot {split_info["n_spots"]:,})')
print(f'边数      : {data.edge_index.shape[1]:,}')
print(f'特征维度  : {data.x.shape[1]}')
print(f'训练/验证 : {data.train_mask.sum()} / {data.val_mask.sum()}')
print('预处理完成 ✓')

In [ ]:
# ============================================================
# Cell 4a: 初始化三种 GNN 模型（每次都执行，构建模型结构）
# ============================================================
from models_amp import GCN_AMP, GraphSAGE_AMP, GAT_AMP
from utils import run_experiment

in_dim = data.x.shape[1]
device = torch.device(PARAMS['device'])

model_configs = {
    'GCN': GCN_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GraphSAGE': GraphSAGE_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
    'GAT': GAT_AMP(
        in_channels     = in_dim,
        hidden_channels = PARAMS['hidden_dim'],
        out_channels    = n_classes,
        proj_dim        = PARAMS['proj_dim'],
        dropout         = PARAMS['dropout'],
    ),
}
print('模型参数量:')
for name, model in model_configs.items():
    n_params = sum(p.numel() for p in model.parameters())
    print(f'  {name}: {n_params:,}')

In [ ]:
# ============================================================
# Cell 4b: 训练 GNN（或从已保存权重加载）
# 缓存策略：
#   {model}_best.pt + gnn_results_summary.pkl 均存在
#                → 直接加载权重，跳过训练
#   任一不存在   → 重新训练，训练后保存
# FORCE['train'] = True 可强制重新训练
# ============================================================

def _weight_file(name):
    return os.path.join(PATHS['output']['models'], f'{name}_best.pt')

ALL_WEIGHTS_OK = (
    all(os.path.exists(_weight_file(n)) for n in model_configs)
    and os.path.exists(RESULTS_SUMMARY_FILE)
    and not FORCE['train']
)

if ALL_WEIGHTS_OK:
    print('[Cache HIT] 加载已训练模型权重...')
    with open(RESULTS_SUMMARY_FILE, 'rb') as f:
        results_summary = pickle.load(f)

    gnn_results = {}
    for name, model in model_configs.items():
        state = torch.load(_weight_file(name),
                           map_location='cpu', weights_only=True)
        model.load_state_dict(state)
        model.eval()
        gnn_results[name] = {
            'model'       : model,
            'best_val_f1' : results_summary[name]['best_val_f1'],
            'best_val_acc': results_summary[name].get('best_val_acc', 0.0),
            'best_epoch'  : results_summary[name].get('best_epoch', -1),
        }
        print(f'  {name:<12} Val F1={results_summary[name]["best_val_f1"]:.4f}')
    print('加载完成 ✓')

else:
    if FORCE['train']:
        print('[强制重训] FORCE["train"]=True，忽略已有权重...')
    else:
        print('[Cache MISS] 未找到完整训练权重，开始训练...')

    gnn_results = {}
    for model_name, model in model_configs.items():
        print(f'\n{"="*55}')
        print(f'  训练 {model_name}')
        print(f'{"="*55}')
        result = run_experiment(
            model         = model,
            data          = data,
            class_weights = class_weights,
            n_classes     = n_classes,
            device        = device,
            params        = PARAMS,
            model_name    = model_name,
            save_dir      = PATHS['output']['models'],
        )
        gnn_results[model_name] = result
        print(f'  Best Val F1: {result["best_val_f1"]:.4f}')

    # 保存训练指标摘要（权重由 run_experiment 保存）
    results_summary = {
        name: {
            'best_val_f1' : res['best_val_f1'],
            'best_val_acc': res.get('best_val_acc', 0.0),
            'best_epoch'  : res.get('best_epoch', -1),
        }
        for name, res in gnn_results.items()
    }
    with open(RESULTS_SUMMARY_FILE, 'wb') as f:
        pickle.dump(results_summary, f)
    print(f'\n✅ 训练指标已保存 → {RESULTS_SUMMARY_FILE}')
    print('   下次运行将直接加载权重，跳过训练')

print('\n模型对比：')
for name, res in gnn_results.items():
    print(f'  {name:<12} Val F1={res["best_val_f1"]:.4f}')

In [ ]:
# ============================================================
# Cell 5: 推断所有节点概率 + scRNA 验证集评估
# 缓存策略：
#   spot_probas_all.pkl 存在 → 直接加载
#   不存在              → 前向推断并保存
# FORCE['eval'] = True 可强制重新推断
# ============================================================
from eval import compute_metrics

val_idx  = split_info['val_idx']
val_true = flex_labels[val_idx]
n_scrna  = split_info['n_scrna']

EVAL_OK = os.path.exists(SPOT_PROBAS_FILE) and not FORCE['eval']

if EVAL_OK:
    print('[Cache HIT] 加载预测概率缓存...')
    with open(SPOT_PROBAS_FILE, 'rb') as f:
        probas_cache = pickle.load(f)
    val_preds   = probas_cache['val_preds']
    spot_probas = probas_cache['spot_probas']

else:
    if FORCE['eval']:
        print('[强制重算] FORCE["eval"]=True，重新推断...')
    else:
        print('[Cache MISS] 前向推断所有节点...')

    val_preds   = {}
    spot_probas = {}
    data_cpu    = data.cpu()

    for name, res in gnn_results.items():
        model_eval = res['model'].to('cpu').eval()
        with torch.no_grad():
            log_probs = model_eval(data_cpu)
            proba_all = torch.softmax(log_probs, dim=1).numpy()
        val_preds[name]   = proba_all[val_idx].argmax(axis=1)
        spot_probas[name] = proba_all[n_scrna:]
        print(f'  {name} 推断完成')

    # 保存概率缓存
    with open(SPOT_PROBAS_FILE, 'wb') as f:
        pickle.dump({'val_preds': val_preds,
                     'spot_probas': spot_probas}, f)
    print(f'✅ 预测概率已保存 → {SPOT_PROBAS_FILE}')

# ── 评估表格 ─────────────────────────────────────────────────
print('\n' + '=' * 65)
print('  scRNA 验证集评估（Ground Truth = 真实细胞类型标签）')
print('=' * 65)
print(f'{"Method":<14} {"Acc":>7} {"F1-Mac":>8} {"F1-Wei":>8} {"Kappa":>8}')
print('-' * 65)
for method, y_pred in val_preds.items():
    m = compute_metrics(val_true, y_pred, n_classes=n_classes)
    print(f'{method:<14} {m["accuracy"]:>7.4f} {m["f1_macro"]:>8.4f} '
          f'{m["f1_weighted"]:>8.4f} {m["kappa"]:>8.4f}')
print('=' * 65)

In [ ]:
# ============================================================
# Cell 6: 保存 spot 级别预测结果（CSV）
# 每次运行都保存（覆盖），保证与最新模型一致
# ============================================================
best_name = max(gnn_results, key=lambda n: gnn_results[n]['best_val_f1'])
print(f'最优模型: {best_name}  Val F1={gnn_results[best_name]["best_val_f1"]:.4f}')

best_proba      = spot_probas[best_name]
best_pred_idx   = best_proba.argmax(axis=1)
best_pred_label = [cell_types_list[i] for i in best_pred_idx]
best_confidence = best_proba.max(axis=1)

spot_df = pd.DataFrame({
    'x'         : spot_coords[:, 0],
    'y'         : spot_coords[:, 1],
    'pred_idx'  : best_pred_idx,
    'pred_label': best_pred_label,
    'confidence': best_confidence,
})
out_csv = f'{PATHS["output"]["predictions"]}/spot_predictions_{best_name}.csv'
spot_df.to_csv(out_csv, index=False)
print(f'Spot 预测已保存: {out_csv}  ({len(spot_df):,} spots)')

for name, proba in spot_probas.items():
    df_all = pd.DataFrame(proba,
        columns=[f'prob_{c}' for c in cell_types_list])
    df_all['x']          = spot_coords[:, 0]
    df_all['y']          = spot_coords[:, 1]
    df_all['pred_label'] = [cell_types_list[i]
                            for i in proba.argmax(axis=1)]
    df_all.to_csv(
        f'{PATHS["output"]["predictions"]}/spot_predictions_{name}_full.csv',
        index=False)
print('所有模型 spot 预测已保存 ✓')

In [ ]:
# ============================================================
# Cell 7: 空间可视化
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, len(gnn_results),
                         figsize=(7*len(gnn_results), 7))
if len(gnn_results) == 1:
    axes = [axes]

cmap   = plt.cm.get_cmap('tab20', n_classes)
colors = {i: cmap(i) for i in range(n_classes)}

for ax, (model_name, proba) in zip(axes, spot_probas.items()):
    pred_idx = proba.argmax(axis=1)
    c_vals   = [colors[i] for i in pred_idx]
    ax.scatter(spot_coords[:, 0], spot_coords[:, 1],
               c=c_vals, s=0.3, alpha=0.5, rasterized=True)
    ax.set_aspect('equal')
    ax.set_title(
        f'{model_name}\nVal F1={gnn_results[model_name]["best_val_f1"]:.3f}',
        fontsize=12)
    ax.set_xlabel('x (μm)')
    ax.set_ylabel('y (μm)')

patches = [mpatches.Patch(color=colors[i], label=cell_types_list[i])
           for i in range(n_classes)]
fig.legend(handles=patches, bbox_to_anchor=(1.01, 0.9),
           loc='upper left', fontsize=8, framealpha=0.8)
plt.suptitle(
    f'GNN Spot-level Cell Type Predictions\n'
    f'Kidney IgAN Xenium | bin={PARAMS["bin_size"]}μm | '
    f'No Cell Segmentation Required',
    fontsize=13, y=1.02)
plt.tight_layout()
out_fig = f'{PATHS["output"]["plots"]}/spot_spatial_all_models.png'
plt.savefig(out_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f'空间图已保存: {out_fig}')

In [ ]:
# ============================================================
# Cell 8: Moran's I 空间自相关
# ============================================================
from topact import TopACT

best_pred_int = spot_probas[best_name].argmax(axis=1)

print(f"计算全局 Moran's I（{best_name}）...")
moran_global = TopACT.morans_i(
    labels      = best_pred_int,
    coords      = spot_coords,
    n_neighbors = 10,
)
print(f"全局 Moran's I = {moran_global['I']:.4f}"
      f" (z={moran_global['z_score']:.2f}, p={moran_global['p_value']:.2e})")

print(f"\n各细胞类型 Moran's I:")
per_class = TopACT.per_class_morans_i(
    labels      = best_pred_int,
    coords      = spot_coords,
    cell_types  = cell_types_list,
    n_neighbors = 10,
)
for ct, vals in sorted(per_class.items(), key=lambda x: -x[1]['I']):
    print(f'  {ct:<12} I={vals["I"]:>7.4f}  p={vals["p_value"]:.2e}')